In [1]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime
from nemosis import static_table
from datetime import timedelta

raw_data_cache = '/Volumes/T7/NEMO-misc'

In [2]:
# # Join DUIDs with firm names and further info
# generator_info_df = static_table(table_name='Generators and Scheduled Loads', 
#                               raw_data_location=raw_data_cache,
#                               update_static_file=False)
# generator_info_df

In [3]:
# # Join generator info with volume bid table on DUIDs
# def match_generators_with_info(input_dir, output_dir, raw_data_cache):
#     """
#     Match generator DUIDs in filtered bid data with generator information
#     to add details about stations, firms, and states.
    
#     Parameters:
#     -----------
#     input_dir : str
#         Directory containing filtered parquet files (bid-volume-filtered-3)
#     output_dir : str
#         Directory where enriched files will be saved
#     raw_data_cache : str
#         Path to the raw NEMOSIS data cache for fetching generator information
#     """
#     print(f"Starting generator information matching process...")
    
#     # Get list of all parquet files in the input directory
#     file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    
#     # Sort the file list to ensure consistent processing order
#     file_list.sort()
    
#     print(f"Found {len(file_list)} parquet files to process")
    
#     # Process each file
#     total_files = len(file_list)
#     for idx, file_path in enumerate(file_list, start=1):
#         file_name = os.path.basename(file_path)
#         print(f"Processing file {idx}/{total_files}: {file_name}")
        
#         try:
#             # Read the parquet file
#             df = pd.read_parquet(file_path)
            
#             # Check that DUID column exists
#             if 'DUID' not in df.columns:
#                 print(f"  - Warning: DUID column not found in {file_name}, skipping file")
#                 continue
            
#             # Count unique DUIDs before merging
#             original_duids = df['DUID'].unique()
#             print(f"  - File contains {len(original_duids)} unique DUIDs")
            
#             # Merge with generator information
#             enriched_df = pd.merge(
#                 df, 
#                 generator_info_df, 
#                 on='DUID', 
#                 how='left'
#             )
            
#             # Check if there are any DUIDs that didn't match
#             matched_duids = enriched_df.dropna(subset=['Station Name']).copy()
#             unmatched_count = len(df) - len(matched_duids)
            
#             if unmatched_count > 0:
#                 print(f"  - Warning: {unmatched_count} rows ({unmatched_count/len(df)*100:.1f}%) have DUIDs not found in generator info")
#                 unmatched_duids = enriched_df[enriched_df['Station Name'].isna()]['DUID'].unique()
#                 print(f"  - Sample of unmatched DUIDs: {list(unmatched_duids)[:5]}")
            
#             # Construct output file path
#             output_file_path = os.path.join(output_dir, file_name)
            
#             # Save the enriched DataFrame to a new parquet file
#             enriched_df.to_parquet(output_file_path, index=False)
#             print(f"  - Saved enriched data to {output_file_path}")
            
#         except Exception as e:
#             print(f"  - Error processing {file_name}: {str(e)}")
    
#     print("\nProcessing complete!")
#     print(f"Enriched data saved to {output_dir}")
    

# if __name__ == "__main__":
#     # Directories
#     input_directory = "/Volumes/T7/bid-volume-filtered-3"  # 18:05:00 filtered data
#     output_directory = "/Volumes/T7/bid-volume-enriched"   # Where to save enriched data
#     raw_data_cache = "/Volumes/T7/NEMO-misc"               # Path to NEMOSIS cache
    
#     # Match generator information with DUID
#     match_generators_with_info(
#         input_dir=input_directory,
#         output_dir=output_directory,
#         raw_data_cache=raw_data_cache
#     )

In [4]:
import glob
import os
import dask.dataframe as dd

directory = "/Volumes/T7/bid-volume-enriched/"
all_files = glob.glob(os.path.join(directory, "*.parquet"))

# Filter out files that start with '._'
parquet_files = [f for f in all_files if not os.path.basename(f).startswith("._")]

# Now read only the valid parquet files
ddf = dd.read_parquet(parquet_files)
pdf = ddf.compute()

In [5]:
pdf.head()

,SETTLEMENTDATE,DUID,BIDTYPE,OFFERDATE,MAXAVAIL,ENABLEMENTMIN,ENABLEMENTMAX,LOWBREAKPOINT,HIGHBREAKPOINT,BANDAVAIL1,...,Station Name,Region,Dispatch Type,Category,Classification,Fuel Source - Primary,Fuel Source - Descriptor,Technology Type - Primary,Technology Type - Descriptor,Aggregation
0,2009/07/01 00:00:00,BASTYAN,LOWERREG,2009/07/01 15:01:02,26,25,78,51,78,0,...,Bastyan Power Station,TAS1,Generating Unit,Market,Scheduled,Hydro,Water,Renewable,Hydro - Gravity,Y
1,2009/07/01 00:00:00,BASTYAN,RAISEREG,2009/07/01 15:01:02,26,0,78,0,52,0,...,Bastyan Power Station,TAS1,Generating Unit,Market,Scheduled,Hydro,Water,Renewable,Hydro - Gravity,Y
2,2009/07/01 00:00:00,BELLBAY1,LOWERREG,2005/04/21 14:31:20,0,36,120,49,120,3,...,None,None,None,None,None,None,None,None,None,None
3,2009/07/01 00:00:00,BELLBAY1,RAISEREG,2005/04/21 14:31:20,0,36,115,36,115,3,...,None,None,None,None,None,None,None,None,None,None
4,2009/07/01 00:00:00,BELLBAY2,LOWERREG,2005/04/21 14:31:20,0,36,120,49,120,3,...,None,None,None,None,None,None,None,None,None,None


In [8]:
# From the previous code, we have transferred from using dask to pandas because we have created a dataframe by calling .compute()
# Therefore, we can now use pandas to melt the dataframe

# 1) Identify the columns to melt
band_cols = [f"BANDAVAIL{i}" for i in range(1, 11)]

# 2) Everything else will be an id_var
id_vars_cols = [col for col in pdf.columns if col not in band_cols]

# 3) Melt the DataFrame
df_long = pd.melt(
    pdf,
    id_vars=id_vars_cols,       # all columns except the band columns
    value_vars=band_cols,       # just the band columns
    var_name="BIDBAND",
    value_name="BIDVOLUME"
)

# 4) Convert BIDBAND (e.g. "BANDAVAIL1") to numeric
df_long["BIDBAND"] = (
    df_long["BIDBAND"].str.extract(r"(\d+)")
    .astype(int)
)

df_long.head()

#Checking all merging columns have the same data type
df_long["INTERVAL_DATETIME"] = pd.to_datetime(df_long["INTERVAL_DATETIME"])


df_long["DUID"] = df_long["DUID"].astype(str)

df_long["BIDTYPE"] = df_long["BIDTYPE"].astype(str)


df_long["BIDBAND"] = df_long["BIDBAND"].astype(int)


# Save df_long to a single Parquet file on disk
output_path = "/Volumes/T7/bid-volume-melted.parquet"
df_long.to_parquet(output_path, index=False)


print(f"Saved melted DataFrame to {output_path}")

Saved melted DataFrame to /Volumes/T7/bid-volume-melted.parquet


In [11]:
# We do the same with price bids

directory = "/Volumes/T7/bid-price-filtered"
all_files = glob.glob(os.path.join(directory, "*.parquet"))

# Filter out files that start with '._'
parquet_files = [f for f in all_files if not os.path.basename(f).startswith("._")]

# Now read only the valid parquet files
price_bids = dd.read_parquet(parquet_files)
price_bids_df = price_bids.compute()

# 1) Identify the columns to melt
band_cols = [f"PRICEBAND{i}" for i in range(1, 11)]

# 2) Everything else will be an id_var
id_vars_cols = [col for col in price_bids_df.columns if col not in band_cols]

# 3) Melt the DataFrame
price_df_long = pd.melt(
    price_bids_df,
    id_vars=id_vars_cols,       # all columns except the band columns
    value_vars=band_cols,       # just the band columns
    var_name="BIDBAND",
    value_name="BIDPRICE"
)

# 4) Convert BIDBAND (e.g. "BANDAVAIL1") to numeric
price_df_long ["BIDBAND"] = (
    price_df_long ["BIDBAND"].str.extract(r"(\d+)")
    .astype(int)
)

price_df_long.head()

price_df_long["BIDBAND"] = price_df_long["BIDBAND"].astype(int)
price_df_long["BIDTYPE"] = price_df_long["BIDTYPE"].astype(str)
price_df_long["DUID"] = price_df_long["DUID"].astype(str)
price_df_long["SETTLEMENTDATE"] = pd.to_datetime(price_df_long["SETTLEMENTDATE"])
price_df_long["APPLICABLEFROM"] = price_df_long["SETTLEMENTDATE"] + timedelta(hours=4, minutes=5)
price_df_long["APPLICABLEFROM"] = pd.to_datetime(price_df_long["APPLICABLEFROM"])


price_df_long.head()

# Save df_long to a single Parquet file on disk
output_path = "/Volumes/T7/bid-price-melted.parquet"
price_df_long.to_parquet(output_path, index=False)


In [ ]:
df_long = df_long.sort_values(["DUID", "BIDTYPE", "BIDBAND", "SETTLEMENTDATE"])
price_df_long = price_df_long.sort_values(["DUID", "BIDTYPE", "BIDBAND", "SETTLEMENTDATE"])

In [3]:
import pandas as pd

# 1) Read the Parquet files directly into Pandas
volume_pdf = pd.read_parquet('/Volumes/T7/melted-bids/bid-volume-melted.parquet')
price_pdf = pd.read_parquet('/Volumes/T7/melted-bids/bid-price-melted.parquet')

# 2) Convert SETTLEMENTDATE to datetime if not already
volume_pdf["SETTLEMENTDATE"] = pd.to_datetime(volume_pdf["SETTLEMENTDATE"])
price_pdf["SETTLEMENTDATE"] = pd.to_datetime(price_pdf["SETTLEMENTDATE"])

# 3) Ensure DUID etc. have the same dtype in both
volume_pdf["DUID"] = volume_pdf["DUID"].astype(str)
price_pdf["DUID"] = price_pdf["DUID"].astype(str)

# 4) Merge on the common columns
merged_pdf = pd.merge(
    volume_pdf,
    price_pdf,
    on=["SETTLEMENTDATE","DUID","BIDTYPE","BIDBAND"],
    how="inner"
)

# 5) merged_pdf is now a plain Pandas DataFrame
print(merged_pdf.head())
print(len(merged_pdf))

  SETTLEMENTDATE      DUID   BIDTYPE          OFFERDATE_x MAXAVAIL  \
0     2009-07-01   BASTYAN  LOWERREG  2009/07/01 15:01:02       26   
1     2009-07-01   BASTYAN  RAISEREG  2009/07/01 15:01:02       26   
2     2009-07-01  BELLBAY1  LOWERREG  2005/04/21 14:31:20        0   
3     2009-07-01  BELLBAY1  RAISEREG  2005/04/21 14:31:20        0   
4     2009-07-01  BELLBAY2  LOWERREG  2005/04/21 14:31:20        0   

  ENABLEMENTMIN ENABLEMENTMAX LOWBREAKPOINT HIGHBREAKPOINT  \
0            25            78            51             78   
1             0            78             0             52   
2            36           120            49            120   
3            36           115            36            115   
4            36           120            49            120   

    INTERVAL_DATETIME  ... BIDVOLUME          OFFERDATE_y VERSIONNO  \
0 2009-07-01 18:05:00  ...         0  2009/06/30 12:10:15        63   
1 2009-07-01 18:05:00  ...         0  2009/06/30 12:10:15       

In [4]:
output_file = "/Volumes/T7/melted-bids/bid-merged.parquet"
merged_pdf.to_parquet(output_file, index=False)

print(f"Saved merged DataFrame to {output_file}")

Saved merged DataFrame to /Volumes/T7/melted-bids/bid-merged.parquet


In [ ]:
# import dask.dataframe as dd

# volume_ddf = dd.read_parquet('/Volumes/T7/melted-bids/bid-volume-melted.parquet')
# price_ddf = dd.read_parquet('/Volumes/T7/melted-bids/bid-price-melted.parquet')

# # Force a simple RangeIndex
# volume_ddf = volume_ddf.reset_index(drop=True)
# price_ddf = price_ddf.reset_index(drop=True)

# # Convert SETTLEMENTDATE to datetime if needed
# volume_ddf["SETTLEMENTDATE"] = dd.to_datetime(volume_ddf["SETTLEMENTDATE"])
# price_ddf["SETTLEMENTDATE"] = dd.to_datetime(price_ddf["SETTLEMENTDATE"])

# # Force DUID etc. to be strings/ints consistently
# volume_ddf["DUID"] = volume_ddf["DUID"].astype(str)
# price_ddf["DUID"] = price_ddf["DUID"].astype(str)

# # Merge
# merged_ddf = volume_ddf.merge(
#     price_ddf,
#     on=["SETTLEMENTDATE","DUID","BIDTYPE","BIDBAND"],
#     how="inner"
# )

# merged_pdf = merged_ddf.compute()  # Should now work

AttributeError: 'Index' object has no attribute '_get_attributes_dict'

In [ ]:
# import os
# import dask.dataframe as dd
# import glob

# def load_all_parquet_dask(parquet_dir):
#     # Identify all Parquet files
#     all_files = glob.glob(os.path.join(parquet_dir, "*.parquet"))
#     # Filter out '._' hidden files
#     parquet_files = [f for f in all_files if not os.path.basename(f).startswith("._")]
#     # Read them into a Dask DataFrame
#     return dd.read_parquet(parquet_files)

# def main():
#     bid_volume_path = '/Volumes/T7/bid-volume-enriched'
#     bid_price_path = '/Volumes/T7/bid-price-filtered'
    
#     print("Loading volume data with Dask...")
#     volume_ddf = load_all_parquet_dask(bid_volume_path)
#     print(volume_ddf.head())  # just to show some data

#     print("Loading price data with Dask...")
#     price_ddf = load_all_parquet_dask(bid_price_path)
#     print(price_ddf.head())  # just to show some data

#     merged_ddf = volume_ddf.merge(
#     price_ddf,
#     on=["SETTLEMENTDATE", "DUID", "BIDTYPE"],
#     how="inner")

#     merged_pdf = merged_ddf.compute()
#     print(f"Merged DataFrame in memory: {merged_pdf.shape}")

#     # Otherwise, you can save the merged result to disk in Parquet or CSV:
#     # (But be aware that saving to CSV with Dask will create multiple partitions
#     # unless you do `coalesce` or `repartition(npartitions=1)`.)
#     output_dir = "/Volumes/T7/complete-bids"
#     merged_ddf.to_parquet(output_dir, overwrite=True)
#     print(f"Saved merged data to {output_dir}")

# if __name__ == "__main__":
#     main()